<a href="https://colab.research.google.com/github/jvillagra66-oss/Ml-Health/blob/Main-colab/Ml_salud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto de machine leargning sobre salud

# 🏥 Proyecto de Predicción de Enfermedades Cardíacas (UCI)

Este repositorio contiene un flujo de trabajo completo para el análisis exploratorio de datos (EDA), la limpieza y el preprocesamiento de un conjunto de datos sobre enfermedades cardíacas proveniente de la **UCI (University of California, Irvine)**.

El objetivo de este proyecto es preparar y estructurar de manera óptima las variables clínicas y demográficas de los pacientes para entrenar modelos predictivos capaces de detectar la presencia o ausencia de afecciones cardíacas.

---

## 📋 Estructura y Flujo del Proyecto

### 1. Conexión e Importación de Datos
* **Entorno:** Google Colab con montaje en Google Drive para la gestión de archivos.
* **Librerías principales:** `pandas`, `numpy`, `matplotlib`, `seaborn` y componentes de `scikit-learn`.
* **Conjunto de datos:** Un dataset de 920 registros clínicos y 16 columnas iniciales.

### 2. Análisis Exploratorio de Datos (EDA)
* **Auditoría de Datos:** Inspección de las dimensiones, tipos de datos (`df.info()`) y distribución estadística de las variables numéricas (`df.describe()`).
* **Visualización de la Variable Objetivo:**
  * **Multiclase (`num`):** Se analizó la distribución original del diagnóstico (valores de `0` a `4`), donde `0` representa ausencia de enfermedad y `1-4` diferentes niveles de gravedad.
  * **Binaria (`enfermo`):** Se transformó la variable a un problema de clasificación binaria (`1` si presenta enfermedad, `0` si no la presenta) para simplificar la detección preventiva.
* **Análisis de Valores Faltantes:** Visualización gráfica mediante un diagrama de barras (`sns.barplot`) para identificar qué variables presentaban mayores vacíos (por ejemplo, `ca`, `thal` y `slope` con mayor cantidad de nulos).

### 3. Preprocesamiento Automatizado (Pipelines)
Se estructuró un flujo de transformación robusto utilizando `ColumnTransformer` de scikit-learn:
* **Variables Numéricas:** Imputación por la media y escalado estándar (`StandardScaler`).
* **Variables Categóricas:** Imputación de nulos por la moda (el valor más frecuente) y codificación con `OneHotEncoder`.
* *Nota de Corrección:* Se solucionó en caliente un error de nombres de columna corrigiendo el descriptor de la variable `'trestbs'` por `'trestbps'` (presión arterial en reposo).

### 4. Normalización Manual y Codificación Alternativa
Como alternativa de ingeniería de características, se generó un flujo personalizado sobre un set `df_model`:
* **Imputación de Nulos:** Los nulos en las columnas numéricas críticas `'trestbps'` y `'chol'` (colesterol) se rellenaron con su mediana respectiva para evitar sesgos por valores extremos.
* **Escalado MinMax:** Se escalaron las variables de `'age'` (edad), `'trestbps'` y `'chol'` para que sus valores queden estrictamente entre `0` y `1`.
* **Codificación de Categóricas:** Aplicación de `pd.get_dummies` de forma manual sobre `'sex'` (género) y `'cp'` (tipo de dolor de pecho).

### 5. Consolidación y Limpieza Final
* Se fusionaron las variables procesadas de características (`x`) con nuestra etiqueta objetivo (`y`).
* Se realizó una limpieza final eliminando cualquier fila residual con valores faltantes mediante un `dropna()` global para asegurar la calidad de la entrada.
* **Resultado:** Un dataset final con variables independientes `x` y etiquetas `y` verificado con `0%` de valores nulos.

### 6. División de Datos (Train-Test Split)
* Se dividieron los datos limpios y normalizados de manera aleatoria bajo una proporción estándar del **80% para entrenamiento** (`x_train`, `y_train`) y **20% para evaluación/prueba** (`x_test`, `y_test`).
* Se utilizó una semilla (`random_state=42`) para garantizar la reproducibilidad del experimento en etapas posteriores de entrenamiento de modelos predictivos.

```markdown
## Introducción

El infarto agudo de miocardio y las enfermedades cardiovasculares representan una de las principales causas de morbimortalidad a nivel global. La detección temprana y la identificación de factores de riesgo clave son fundamentales para la intervención médica oportuna y la mejora en la expectativa y calidad de vida de los pacientes.

En este proyecto de **Ciencia de Datos y Aprendizaje Automático**, utilizaremos el reconocido dataset de enfermedades cardíacas de la base de datos de la **UCI (University of California, Irvine)**. A través de este análisis y modelado predictivo, buscaremos:

1. **Explorar y Comprender los Datos:** Analizar variables clínicas y demográficas (tales como edad, sexo, niveles de colesterol, presión arterial y resultados de electrocardiogramas) para identificar patrones asociados a la presencia de patologías cardíacas.
2. **Tratamiento y Limpieza de Datos:** Gestionar los valores nulos y estructurar la variable objetivo para enfoques de clasificación tanto multiclase como binaria (presencia o ausencia de enfermedad).
3. **Desarrollar Modelos Predictivos:** Entrenar algoritmos de Machine Learning capaces de estimar el riesgo o presencia de una afección cardíaca en función del historial clínico del paciente.

Este enfoque preventivo basado en datos aspira a servir como una herramienta de apoyo en la toma de decisiones clínicas, facilitando diagnósticos más rápidos y precisos.
```

In [ ]:
#Importamos las librerías que utilizaremos a lo largo del proyecto
import pandas as pd
import numpy as np

# Para visualizaciones
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

# Configuraciones generales
pd.set_option('display.max_columns', None)  # Mostrar todas las columnas
sns.set(style="whitegrid")

# Cargamos el dataset
df = pd.read_csv("/content/drive/MyDrive/Proyecto de ciencia de dato de salud/heart_disease_uci.csv")

# Mostramos las primeras filas
df.head()

# Se crea el data frame

In [ ]:
df.info()

# Describcion del data frame

In [ ]:
df.describe()

# Distribución de la variable objetivo

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='num', palette="pastel")
plt.title("Distribución de la variable objetivo (num)")
plt.xlabel("Valor de 'num' (diagnóstico)")
plt.ylabel("Cantidad de casos")
plt.show()


# Distribucion Binaria y visualizacion

In [ ]:
# Creamos una nueva columna binaria: "enfermo"
df["enfermo"] = df["num"].apply(lambda x: 1 if x > 0 else 0)

# Visualizamos la distribucion
plt.figure(figsize=(5,3))
sns.countplot(data=df, x="enfermo" , palette="pastel")
plt.title("Distribucion binaria: presencia de enfermedad cadiaca")
plt.xlabel("Enfermo(1 = si, 0 = no)")
plt.ylabel("Cantidad de casos")
plt.show()

# Calculo y cantidad de valores nulos como visualizacion

In [ ]:
# Calculamos cantidad de valores nulos por columna
valores_nulos = df.isnull().sum()
valores_nulos = valores_nulos[valores_nulos > 0].sort_values(ascending=False)

# Visualizacion
plt.figure(figsize=(8,5))
sns.barplot(x=valores_nulos.values, y=valores_nulos.index, palette="rocket")
plt.title("Cantidad de valores faltantes por variables")
plt.xlabel("Cantidad de valores nulos")
plt.ylabel("Variables")
plt.show()


# Separamos las columnas segun su tipo de datos

In [ ]:
variables_numericas = df.select_dtypes(include=["int64",]).columns.tolist()
variables_categoricas = df.select_dtypes(include="object").columns.tolist()

# Mostramos las lista (excluyente "id" y "num" que no usaremos como predictores )

In [ ]:
print("Variables numéricas:")
print([col for col in variables_numericas if col not in ['id', 'num']])

print("\nVariables categóricas:")
print(variables_categoricas)

# Importamos librerias

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Variables que usaremos como predictoras (excluimos id , dataset, num , enfermo)

In [ ]:
variables_numericas = ["age" , "trestbs" , "chol" , "thalch" , "oldpeak" , "ca"]
variables_categoricas = ["sex" , "cp" , "fbs" , "restecg" , "exang" , "slope" , "thal"]

# Pipeline para variables numericas: imputacion por media + escalado

In [ ]:
variables_numericos = Pipeline(steps=[
    ("imputacion" , SimpleImputer(strategy="mean")),
    ("escalando", StandardScaler())
])

# Pipeline para variables categoricas : imputacion por moda + codificacion one-hot

In [ ]:
pipeline_categorico = Pipeline(steps=[
 ("imputacion", SimpleImputer(strategy="most_frequent")),
 ("codificacion" , OneHotEncoder(handle_unknown="ignore"))
])

# papiline general de preprocesamiento

In [ ]:
preprocesador = ColumnTransformer(transformers=[
    ('num', variables_numericos, variables_numericas),
    ('cat', pipeline_categorico, variables_categoricas)
])

 # Definimos las columnas que vamos a usar como features

In [ ]:
columnas_a_utilizar = variables_numericas + variables_categoricas

# Aplicamos el pipeline de preprocesamiento a esas columnas

In [ ]:
# Corregimos temporalmente el error de escritura 'trestbs' -> 'trestbps' para poder ejecutar el preprocesador
columnas_corregidas = [col if col != 'trestbs' else 'trestbps' for col in columnas_a_utilizar]
variables_numericas_corregidas = [col if col != 'trestbs' else 'trestbps' for col in variables_numericas]

# Actualizamos el preprocesador con los nombres de columnas correctos
preprocesador.transformers[0] = ('num', variables_numericos, variables_numericas_corregidas)

# Aplicamos la transformación
X_transformado = preprocesador.fit_transform(df[columnas_corregidas])

 #Mostramos la forma del resultado y un pequeño fragmento

In [ ]:
print(f"Forma del array transformado: {X_transformado.shape}")


# Si queremos ver algunas filas como DataFrame:

In [ ]:
X_transformado_df = pd.DataFrame(X_transformado.toarray() if hasattr(X_transformado, "toarray") else X_transformado)
X_transformado_df.head()

# Tipos de datos del data frame

In [ ]:
df.dtypes

# Valores nulos encontrados


In [ ]:
df.isnull().sum()

# Valores duplicados de Datframe

In [ ]:
df.duplicated().sum()

# Datos duplicados por filas y columnas

In [ ]:
# 1. cantidad total de filas identicas duplicados
duplicados_total = df.duplicated().sum()
print(f"filas 100% duplicadas: {duplicados_total}")

# 2. ver una muestra de esas filas duplicadas
df[df.duplicated(keep=False)].head(6)

# Variables categoricas

In [ ]:
# Detectamos variables categoricas y numericas
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Mostrar los resultados
print(f"Variables categoricas ({len(cat_cols)}):")
print(cat_cols)
print("\nVariables numericas ({len(num_cols)})")
print(num_cols)

# Copia del dataframe junto con one-hot

In [ ]:
# Se realiza una copia de Dataframe

df_temp = df.copy()

# Se aplican one-hot Encoding y guardamos solo las nuevas columnas utilizando una columna que sí existe (por ejemplo, 'cp')
cp_dummies = pd.get_dummies(df_temp["cp"], prefix="cp", drop_first=False)

display(cp_dummies)

# Concatenacion del Dataframe

In [ ]:
# Concatenamos el nuevo DataFrame original con las variables dummies correctas
df_temp = pd.concat([df_temp, cp_dummies], axis=1)

# Mostramos las nuevas columnas codificadas de dolor de pecho (cp)
df_temp[["cp_typical angina", "cp_atypical angina", "cp_non-anginal", "cp_asymptomatic"]].head()

# Se aplica el get_dummies

In [ ]:
import pandas as pd

# 1. Aplicamos get_dummies a una columna existente (por ejemplo, 'sex') con la sintaxis corregida
sex_dummies = pd.get_dummies(df["sex"], prefix="sex")

# 2. Imprimimos para verificar
print(sex_dummies.head())

# 3. Integramos las nuevas columnas al DataFrame principal
df = pd.concat([df, sex_dummies], axis=1)

# Se normaliza el DataFrame para usarlo en el modelo

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Copia de respaldo
df_model = df.copy()

# 2. One-hot Encoding para variables categóricas reales de tu dataset (sex y cp)
sex_dummies = pd.get_dummies(df_model["sex"], prefix="sex", drop_first=False)
cp_dummies = pd.get_dummies(df_model["cp"], prefix="cp", drop_first=False)

# 3. Escala MinMax para variables numéricas reales
# Usamos 'trestbps' (presión arterial) y 'chol' (colesterol) que contienen algunos nulos, por lo que primero los imputamos temporalmente con la mediana para poder escalar
df_model["trestbps"] = df_model["trestbps"].fillna(df_model["trestbps"].median())
df_model["chol"] = df_model["chol"].fillna(df_model["chol"].median())

scaler = MinMaxScaler()
df_model[["age_normalizada", "trestbps_normalizada", "chol_normalizada"]] = scaler.fit_transform(df_model[["age", "trestbps", "chol"]])

# 4. Definir variable objetivo "y" (usamos la variable binaria 'enfermo')
y = df_model["enfermo"]

# Concatenamos las columnas numéricas normalizadas y las dummies para crear las características 'x'
columnas_feature = [
    "age_normalizada",
    "trestbps_normalizada",
    "chol_normalizada"
]

x = pd.concat([df_model[columnas_feature], sex_dummies, cp_dummies], axis=1)

# Mostrar las primeras filas de x
x.head()

# Combinamos X e Y para eliminar filas con valores flatantes si los tenemos

In [ ]:
df_modelo_clean = pd.concat([x, y], axis=1).dropna()

# Separamos nuevamente x e y de forma correcta para tu proyecto de salud
y = df_modelo_clean["enfermo"]
x = df_modelo_clean.drop(columns=["enfermo"])

# Verificamos que no queden valores faltantes
print("¿Hay NaN en x?:", x.isna().sum().any())
print("¿Hay NaN en y?:", y.isna().sum().any())

#Dividimos los datos: 80% entrenamiento y 20% prueba

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Verificamos las dimensiones finales
print(f"Datos de entrenamiento:{x_train.shape}")
print(f"Datos de prueba {x_test.shape}")

#Se crean Matriz de confusio para ver datos como el acuracy

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

# 1. Creamos y entrenamos el modelo predictivo que faltaba definir
modelo_surf = LogisticRegression(random_state=42)
modelo_surf.fit(x_train, y_train)

# 2. Realizar predicciones con el umbral por defecto (0.50)
y_pred = modelo_surf.predict(x_test)

# 3. Matriz de Confusión
cm = confusion_matrix(y_test, y_pred, labels=[1, 0])

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', annot_kws={"size": 14},
            xticklabels=['Predice Riesgo (1)', 'Predice Normal (0)'],
            yticklabels=['Real Riesgo (1)', 'Real Normal (0)'])
plt.title("Matriz de Confusión - Umbral Por Defecto (50%)")
plt.show()

# 4. Reporte de métricas
print(classification_report(y_test, y_pred, labels=[1, 0], target_names=['Riesgo (1)', 'Normal (0)']))